<link rel="stylesheet" href="../../styles.css">
<div class="note">

<h1>Overfitting</h1>

<img src="../../assets/dudl-04-overfitting.png">
<p>A model is overfitted if it fits tightly the training data and fails to fit new data. Overfitted models fail to encapsulate the underlying nature of the domain, instead it encapsulates the noise too heavily.</p>
<p>How can we tell if we are overfitting the model? If the model is 1 or 2 dimensional, we can visualize it, but in higher dimensions it is not practical, or even possible. In these cases we need to use statistical analysis:
<ul>
<li>cross-validation</li>
<li>regularization</li>
</ul>
Overfirring can be caused by the researcher, more specifically by the criteria the researcher uses for cleaning and organizing the data-
</p>
<h2>Cross-validation</h2>
<p>Cross-validation refers to the practice of separating the data to training, development and test datasets.<br>We train the data on the training set, and then use the development and test sets for validating the model.<br>Typically 80-90% of the data is the training data, the rest is the test sets.</p>
<p>After the training is done, we try it on the dev set without any backpropagation. If we are not satisfied, we tune the model and start again. Once we are satisfied we apply the model to the test set.</p>
<p>K-fold cross-validation: We separate a training and a test set. After training and testing the model, we choose another chunk of the data for test set, and train the model again. We repeat this "k" times.</p>

<p>When we use cross-validation the underlying assumption is that the training data and the test data are independent and do not correlate.</p>
</div>

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
iris = sns.load_dataset('iris')
data = torch.tensor(iris[iris.columns[0:4]].values).float()

labels = torch.zeros(len(data), dtype=torch.long)
labels[iris.species=='versicolor'] = 1
labels[iris.species=='virginica'] = 2

propTraining = 0.8
ntraining = int(len(data)*propTraining)
trainTestBool = np.zeros(len(labels), dtype=bool)
trainTestBool[range(ntraining)] = True

In [17]:
# The average will tell about the ratio of different species in the different sets. The dataset appears to be skewed because the order is not randomized.
print(f'Average of data:          {torch.mean(labels.float())}')
print(f'Average of training data: {torch.mean(labels[trainTestBool].float())}')
print(f'Average of test data:     {torch.mean(labels[~trainTestBool].float())}')

Average of data:          1.0
Average of training data: 0.75
Average of test data:     2.0


In [34]:
itemsToUseForTraining = np.random.choice(range(len(labels)), ntraining, replace=False)
trainTestBool = np.zeros(len(labels), dtype=bool)
trainTestBool[itemsToUseForTraining] = True

print(f'Average of data:          {torch.mean(labels.float())}')
print(f'Average of training data: {torch.mean(labels[trainTestBool].float())}')
print(f'Average of test data:     {torch.mean(labels[~trainTestBool].float())}')

Average of data:          1.0
Average of training data: 0.9750000238418579
Average of test data:     1.100000023841858


In [36]:
ANNiris = nn.Sequential(
    nn.Linear(4, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

lossFunction = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(ANNiris.parameters(), lr=0.01)

In [39]:
epochs = 1_000

losses = torch.zeros(epochs)
ongoingAccuracy = []
for i in range(epochs):
    yHat = ANNiris(data[trainTestBool,:])

    ongoingAccuracy.append(100*torch.mean((torch.argmax(yHat,axis=1) == labels[trainTestBool]).float()))

    loss = lossFunction(yHat, labels[trainTestBool])
    losses[i] = loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [46]:
predictions = ANNiris(data[trainTestBool,:])
trainAccuracy = 100 * torch.mean( (torch.argmax(predictions,axis=1) == labels[trainTestBool]).float() )

predictions = ANNiris(data[~trainTestBool,:])
testAccuracy = 100 * torch.mean( (torch.argmax(predictions,axis=1) == labels[~trainTestBool]).float() )

print(f'trainAccuracy={trainAccuracy} %')
print(f'testAccuracy= {testAccuracy} %')

trainAccuracy=98.33333587646484 %
testAccuracy= 96.66666412353516 %
